In [1]:
from pathlib import Path
import gc
import pandas as pd

RAW = Path("/scratch/sm3664/medicare-fraud/raw/part_b")
PROC = Path("/scratch/sm3664/medicare-fraud/processed/part_b")
CLEAN_DIR = PROC / "clean"
AGG_DIR = PROC / "agg"
for p in (CLEAN_DIR, AGG_DIR):
    p.mkdir(parents=True, exist_ok=True)

YEARS = [2019, 2020, 2021, 2022, 2023]

In [2]:
NUM_COLS = ["Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs", "Avg_Sbmtd_Chrg", "Avg_Mdcr_Pymt_Amt"]
KEEP = ["Rndrng_NPI", "Rndrng_Prvdr_Ent_Cd", "Rndrng_Prvdr_Type", "Place_Of_Srvc", "HCPCS_Cd"] + NUM_COLS
DTYPES = {
    "Rndrng_NPI": "string", "HCPCS_Cd": "string",
    "Rndrng_Prvdr_Ent_Cd": "category", "Rndrng_Prvdr_Type": "category", "Place_Of_Srvc": "category",
    **{c: "float64" for c in NUM_COLS},
}

def clean_year(year):
    name = f"Medicare_Physician_Other_Practitioners_by_Provider_and_Service_{year}"
    d = pd.read_csv(RAW / name / f"{name}.csv", usecols=KEEP, dtype=DTYPES)
    d.insert(1, "year", year)
    # Checks confirmed in notebook 01 - fail loudly if a year ever breaks them
    assert d.isna().sum().sum() == 0, "unexpected nulls"
    assert d["Rndrng_NPI"].str.fullmatch(r"\d{10}").all(), "bad NPI format"
    assert not d.duplicated(["Rndrng_NPI", "HCPCS_Cd", "Place_Of_Srvc"]).any(), "duplicate keys"
    assert d.groupby("Rndrng_NPI")["Rndrng_Prvdr_Type"].nunique().max() == 1, "NPI with >1 provider type"
    assert d.groupby("Rndrng_NPI")["Rndrng_Prvdr_Ent_Cd"].nunique().max() == 1, "NPI with >1 entity code"
    return d

In [3]:
d19 = clean_year(2019)
print(d19.shape)
print(d19.dtypes)
print(f"Memory: {d19.memory_usage(deep=True).sum() / 1e9:.2f} GB")
d19.head()

(10140228, 11)
Rndrng_NPI               string
year                      int64
Rndrng_Prvdr_Ent_Cd    category
Rndrng_Prvdr_Type      category
HCPCS_Cd                 string
Place_Of_Srvc          category
Tot_Benes               float64
Tot_Srvcs               float64
Tot_Bene_Day_Srvcs      float64
Avg_Sbmtd_Chrg          float64
Avg_Mdcr_Pymt_Amt       float64
dtype: object
Memory: 0.83 GB


,Rndrng_NPI,year,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_Type,HCPCS_Cd,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Pymt_Amt
0,1003000126,2019,I,Internal Medicine,99217,F,40.0,40.0,40.0,232.275000,57.870000
1,1003000126,2019,I,Internal Medicine,99220,F,25.0,25.0,25.0,712.800000,148.851600
2,1003000126,2019,I,Internal Medicine,99221,F,24.0,24.0,24.0,320.166667,80.615833
3,1003000126,2019,I,Internal Medicine,99223,F,157.0,158.0,158.0,651.417722,160.190000
4,1003000126,2019,I,Internal Medicine,99232,F,117.0,205.0,205.0,245.614634,58.005561


In [4]:
STATS = ["min", "max", "median", "mean", "sum", "std"]
GROUP = ["Rndrng_NPI", "year", "Rndrng_Prvdr_Type", "Rndrng_Prvdr_Ent_Cd"]

def aggregate_year(d):
    a = d.groupby(GROUP, observed=True)[NUM_COLS].agg(STATS)
    a.columns = [f"{c}_{s}" for c, s in a.columns]
    std_cols = [f"{c}_std" for c in NUM_COLS]
    a[std_cols] = a[std_cols].fillna(0)   # single-row provider: std undefined -> 0, as in paper
    a = a.reset_index()
    assert not a.duplicated(["Rndrng_NPI", "year"]).any(), "more than one row per NPI-year"
    return a

In [5]:
agg19 = aggregate_year(d19)
print(agg19.shape, "| expected rows:", d19["Rndrng_NPI"].nunique())
print("std NaN remaining:", agg19.filter(like="_std").isna().sum().sum())
print("sum < max violations:", (agg19["Tot_Srvcs_sum"] < agg19["Tot_Srvcs_max"]).sum())

npi = agg19["Rndrng_NPI"].iloc[0]
manual = d19.loc[d19["Rndrng_NPI"] == npi, "Avg_Sbmtd_Chrg"].agg(STATS)
auto = agg19.loc[agg19["Rndrng_NPI"] == npi, [f"Avg_Sbmtd_Chrg_{s}" for s in STATS]].iloc[0]
print(pd.DataFrame({"manual": manual.values, "pipeline": auto.values}, index=STATS))

(1093367, 34) | expected rows: 1093367
std NaN remaining: 0
sum < max violations: 0
             manual     pipeline
min      229.475000   229.475000
max      712.800000   712.800000
median   345.236364   345.236364
mean     420.854343   420.854343
sum     3787.689090  3787.689090
std      206.458732   206.458732


In [6]:
del d19, agg19; gc.collect()

for y in YEARS:
    agg_path = AGG_DIR / f"part_b_agg_{y}.parquet"
    if agg_path.exists():
        print(y, "cached, skipping")
        continue
    d = clean_year(y)
    d.to_parquet(CLEAN_DIR / f"part_b_clean_{y}.parquet", index=False)
    a = aggregate_year(d)
    a.to_parquet(agg_path, index=False)
    print(y, "clean:", d.shape, "agg:", a.shape)
    del d, a; gc.collect()

2019 clean: (10140228, 11) agg: (1093367, 34)
2020 clean: (9449361, 11) agg: (1085313, 34)
2021 clean: (9886177, 11) agg: (1123589, 34)
2022 clean: (9755427, 11) agg: (1148873, 34)
2023 clean: (9660647, 11) agg: (1175281, 34)


In [7]:
agg = pd.concat([pd.read_parquet(AGG_DIR / f"part_b_agg_{y}.parquet") for y in YEARS], ignore_index=True)
for c in ["Rndrng_Prvdr_Type", "Rndrng_Prvdr_Ent_Cd"]:
    agg[c] = agg[c].astype("category")

assert not agg.duplicated(["Rndrng_NPI", "year"]).any()
print(agg.shape)
print(agg.groupby("year").size())
print("Unique NPIs across years:", agg["Rndrng_NPI"].nunique())
agg.to_parquet(PROC / "part_b_agg_2019_2023.parquet", index=False)

(5626423, 34)
year
2019    1093367
2020    1085313
2021    1123589
2022    1148873
2023    1175281
dtype: int64
Unique NPIs across years: 1493778
